In [ ]:
#kann gelöscht werden
#Bestimmen der Anzahl von Clustern in den TSV dateien durch MMseqs2

import pandas as pd

# 1.) Lade die TSV ganz ohne Header und benenne alle drei Spalten
df = pd.read_csv(
    "MMseqs2/SEQ_H1_7_clusters.tsv",
    sep="\t",
    header=None, 
    names=["rep","member","Antigen"],
    dtype=str
)

# 2.) Zähle die eindeutigen Cluster-Repräsentanten in Spalte 'rep'
anzahl_cluster = df["rep"].nunique()
print(f"Anzahl Cluster: {anzahl_cluster}")



Anzahl Cluster: 289


In [15]:
#alle relevanten imports:

import pandas as pd
import os
import glob
from Bio import SeqIO
from pathlib import Path
from scipy.stats import chi2_contingency 
import re


In [2]:

seq_regions = ["SEQ_H1", "SEQ_H2", "SEQ_L1", "SEQ_L2", "SEQ_L3"]
cf_regions = ["CF_H1", "CF_H2", "CF_L1", "CF_L2", "CF_L3"]
df = (
    pd.read_csv("data/ab_ag_scalop.tsv", sep="\t")
    .dropna(subset = seq_regions)
    .dropna(subset = cf_regions)
    .drop_duplicates(subset = seq_regions) 
)

antigen_counts = df["antigen_name"].value_counts() # Tabelle aus antigen_names und ihren Häufigkeiten in der Spalte antigen_name
df = df[df["antigen_name"].isin(antigen_counts[antigen_counts >= 5].index)] # Behält nur Zeilen, deren antigen_name mindestens 5-mal vorkommt



In [ ]:
antigen_labels = df["antigen_name"].tolist()

In [7]:
#FASTA erzeugung für CDRs in Subgruppen
#mit PDB und Antigennamen


# Zielordner für die FASTAs
fasta_dir = "data\\MMseqs2\\MMseqs2_Fasta"


os.makedirs(fasta_dir, exist_ok=True)



# Für jede Region: Länge berechnen, nach Länge gruppieren und FASTA-Dateien schreiben
for region in seq_regions:
    length_col = f"{region}_length"
    df[length_col] = df[region].str.len()

    for length, subdf in df.groupby(length_col):
        fasta_path = os.path.join(fasta_dir, f"{region}_{length}.fasta")
        with open(fasta_path, 'w') as out:
            for i, row in enumerate(subdf.itertuples(index=False), start=1):
                seq     = getattr(row, region)
                pdb     = getattr(row, 'pdb')              
                antigen = getattr(row, 'antigen_name').replace(' ', '_')

                header = f"seq_{i} {pdb}|{antigen}"
                out.write(f">{header}\n{seq}\n")


#funktioniert wie es soll

In [ ]:
#Clustern in WSL mit MMSeqs2
#Reine Dokumentation

set -uuo pipefail
# (ohne -e, damit wir Fehler selbst behandeln)

fasta_dir="data/MMseqs2/MMseqs2_Fasta"
outdir="data/MMseqs2/MMseqs2_cluster"
mkdir -p "$outdir"

for fasta in "${fasta_dir}"/SEQ_*_*.fasta; do
  # Wenn im Ordner nichts liegt, abbrechen
  [[ -e "$fasta" ]] || break

  base=$(basename "$fasta" .fasta)     # z.B. "CDR_H1_6"
  region=${base%_*}                    # "CDR_H1"
  length=${base##*_}                   # "6"

  db="${region}_${length}_db"
  clu="${region}_${length}_clu"
  tsv="${outdir}/${region}_${length}_clusters.tsv"

  # 1) FASTA → MMseqs2-Datenbank  
  if ! mmseqs createdb "$fasta" "$db"; then
    continue
  fi

  # 2) Clustern mit Fehlerabfang  
  if ! mmseqs cluster \
        --min-seq-id 0.6 \
        -c 1 \
        --spaced-kmer-mode 0 \
        "$db" "$clu" tmp; then
    rm -rf tmp "$db" "$clu"
    continue
  fi

  # 3) TSV-Export  
  if ! mmseqs createtsv "$db" "$db" "$clu" "$tsv"; then
    rm -rf tmp "$db" "$clu"
    continue
  fi

  # 4) Aufräumen
  rm -rf tmp "$db" "$clu"
done


In [26]:
#Antigenspalte zu cluster daten hinzufügen

# Pfade anpassen
cluster_dir = "data\\MMseqs2\\MMseqs2_cluster"
fasta_dir   = "data\\MMseqs2\\MMseqs2_Fasta"

# Alle Cluster-Dateien finden
cluster_files = glob.glob(os.path.join(cluster_dir, "SEQ_*_clusters.tsv"))

for cluster_file in cluster_files:
    # zugehörige FASTA im anderen Verzeichnis suchen
    base = os.path.basename(cluster_file).replace("_clusters.tsv", ".fasta")
    fasta_file = os.path.join(fasta_dir, base)
    if not os.path.exists(fasta_file):
        continue       # oder: pass, raise FileNotFoundError(...), print(...) etc.

        

    # Mapping seq_id → antigen_name (Text nach '|' im Header)
    id2antigen = {}
    for rec in SeqIO.parse(fasta_file, "fasta"):
        antigen = rec.description.split("|", 1)[1] if "|" in rec.description else ""
        id2antigen[rec.id] = antigen

    # Cluster-TSV einlesen
    df = pd.read_csv(cluster_file, sep="\t", header=None,
                     names=["seq_rep", "seq_member"])

    # Nur Antigen des Members anhängen
    df["antigen_name"] = df["seq_member"].map(id2antigen)

    # Überschreibe die Original-Datei
    df.to_csv(cluster_file, sep="\t", index=False)
    



#funktioniert, wie es soll



In [27]:
#Reihenfolge der Cluster anpassen

from pathlib import Path
import pandas as pd
import re

# 1) Ordner mit den Cluster-Dateien
cluster_folder = Path("data/MMseqs2/MMseqs2_cluster")

# 2) Schleife über alle *_clusters.tsv-Dateien (keine zusätzliche Filename-Sortierung)
for path in cluster_folder.glob("*_clusters.tsv"):
    

    # 3) Header-Zeile separat einlesen
    with open(path, "r") as f:
        header_line = f.readline().rstrip("\n")

    # 4) Rest der Datei in ein DataFrame einlesen
    df = pd.read_csv(
        path,
        sep="\t",
        header=None,
        names=["seq_rep", "seq_member", "antigen_name"],
        dtype=str,
        skiprows=1
    )

    # 5) Nummern extrahieren und als int casten
    #    (\d+ am Ende der Zeichenkette)
    df["rep_num"] = (
        df["seq_rep"]
          .str.extract(r"(\d+)$", expand=False)
          .astype(int)
    )
    df["member_num"] = (
        df["seq_member"]
          .str.extract(r"(\d+)$", expand=False)
          .astype(int)
    )

    # 6) Nach rep_num, dann member_num sortieren
    df_sorted = df.sort_values(
        by=["rep_num", "member_num"],
        ascending=[True, True],
        ignore_index=True,
        na_position="last"
    )

    # 7) Überschreibe die Datei: alter Header + sortierte Zeilen
    with open(path, "w", newline="") as f:
        # alter Header
        f.write(header_line + "\n")
        # nur die drei Original-Spalten, in der neuen Reihenfolge
        df_sorted[["seq_rep", "seq_member", "antigen_name"]] \
            .to_csv(f, sep="\t", header=False, index=False)



#funktioniert auch, nur vorher muss die Antigenspalte hinzugefügt werden

In [31]:
#Proportionstest chi-square für subgruppen cluster



# 1) Alle neuen Subgruppen-TSV-Dateien finden
tsv_files = sorted(glob.glob("data/MMseqs2/MMseqs2_cluster/*_clusters.tsv"))

# 2) Für jede Datei Chi²-Test durchführen
for tsv in tsv_files:
    # a) Region und Länge aus dem Dateinamen extrahieren
    #    "CDR_H1_6_clusters.tsv" → region="CDR_H1", length="6"
    base   = os.path.basename(tsv).removesuffix("_clusters.tsv")
    region, length = base.rsplit("_", 1)
    
    # b) TSV einlesen
    df = pd.read_csv(
        tsv, sep="\t", header=None,
        names=["rep","member","Antigen"], dtype=str
    )
    
    # c) Kontingenztabelle (Antigen × Cluster-Rep)
    contingency = pd.crosstab(df["Antigen"], df["rep"])
    
    # d) Ausgabe
    print(f"\n=== {region} (Länge {length} ) ===")
    print("Kontingenztabelle:")
    print(contingency)
    
    # e) Chi-Quadrat-Test
    chi2, p, dof, expected = chi2_contingency(contingency)
    print(f"\nChi² = {chi2:.4f}, p-Wert = {p:.4e}, df = {dof}")
    
    # f) Erwartete Häufigkeiten unter H₀
    expected_df = pd.DataFrame(
        expected, index=contingency.index, columns=contingency.columns
    )
    print("Erwartete Häufigkeiten (unter H₀):")
    print(expected_df.round(2))



=== SEQ_H1 (Länge 7 ) ===
Kontingenztabelle:
rep                                                 seq_10  seq_100  seq_101  \
Antigen                                                                        
25_kda_ookinete_surface_antigen                          0        0        0   
antigen_name                                             0        0        0   
ch848.10.17_gp120                                        0        0        0   
circumsporozoite_protein                                 0        0        0   
clade_a/e_93th057_hiv-1_gp120_core                       0        0        0   
cytotoxic_t-lymphocyte_protein_4                         0        0        0   
envelope_glycoprotein_b                                  0        0        0   
envelope_glycoprotein_e2                                 0        1        0   
envelope_glycoprotein_gp120                              0        0        0   
envelope_glycoprotein_gp160                              0        0       

In [34]:
#pdb_id hinzufügen zu geclusterte datei

cluster_dir = "data\\MMseqs2\\MMseqs2_cluster"
fasta_dir   = "data\\MMseqs2\\MMseqs2_Fasta"


cluster_files = glob.glob(os.path.join(cluster_dir, "SEQ_*_clusters.tsv"))

for cluster_file in cluster_files:
    # --- 1) FASTA-Mapping aufbauen ---
    base       = os.path.basename(cluster_file).replace("_clusters.tsv", ".fasta")
    fasta_file = os.path.join(fasta_dir, base)
    if not os.path.exists(fasta_file):
        continue

    id2pdb     = {}
    id2antigen = {}
    for rec in SeqIO.parse(fasta_file, "fasta"):
        parts = rec.description.split()
        if len(parts) > 1 and "|" in parts[1]:
            pdb, antigen = parts[1].split("|", 1)
        else:
            pdb, antigen = "", ""
        id2pdb[rec.id]     = pdb
        id2antigen[rec.id] = antigen

    # --- 2) Prüfen, ob TSV schon einen Header hat ---
    with open(cluster_file, 'r') as f:
        first = f.readline().strip().split('\t')
    has_header = first[0] == "seq_rep"

    # --- 3) Nur die ersten beiden Spalten einlesen, Header ggf. überspringen ---
    df = pd.read_csv(
        cluster_file,
        sep="\t",
        header=None,
        names=["seq_rep", "seq_member"],
        usecols=[0, 1],
        skiprows=1 if has_header else 0,
        dtype=str,
        engine="python"
    )

    # --- 4) Neue Spalten hinzufügen ---
    df["antigen_name"] = df["seq_member"].map(id2antigen).fillna("")
    df["pdb_member"]     = df["seq_member"].map(id2pdb).fillna("")

    # --- 5) Kompletten TSV mit ALLEN Spalten und Header zurückschreiben ---
    df.to_csv(cluster_file, sep="\t", index=False)




In [ ]:
#Enno Plakat:

#resultate zeigen, was hat geklappt bzw. was hat nicht so gut geklappt, was hat gar nicht funktioniert, warum hat es nicht funktioniert
#was sind dlie Mimitationen dieses clustering verfahren, warum nicht unbedingt so passend zu unserem Projekt
#sequenz identity und das aligment kurz ansprechen

#für Präsentation, warum MMSeqs2 ausprobiert

In [36]:
#Datei zum Vergleich von Vmeasure erstellen



# Wenn Deine Files in "MMSeqs2/" liegen, hier den Pfad anpassen
BASE_DIR = "data\\MMseqs2\\MMseqs2_cluster"

cdr_regions = ["H1", "H2", "L1", "L2", "L3"]

def load_and_label(region):
    parts = []
    # Suche in MMSeqs2/ nach SEQ_{region}_*_clusters.tsv
    pattern = os.path.join(BASE_DIR, f"SEQ_{region}_*_clusters.tsv")
    for fn in glob.glob(pattern):
        length = os.path.basename(fn).split("_")[2]
        prefix = f"{region}-{length}"
        colname = f"CF_{region}"
        
        df = pd.read_csv(fn, sep="\t", usecols=["seq_rep", "antigen_name", "pdb_member"])
        df[colname] = df["seq_rep"].apply(lambda r: f"{prefix}-{r}")
        parts.append(df[["pdb_member", "antigen_name", colname]])
    
    if not parts:
        return None
    
    out = pd.concat(parts, ignore_index=True).drop_duplicates(
        subset=["pdb_member", "antigen_name"]
    )
    return out.set_index(["pdb_member", "antigen_name"]) #dadurch wird auf beide Spalten indexiert

# 1) Lade alle Regionen
region_dfs = {}
for region in cdr_regions:
    df = load_and_label(region)
    if df is not None:
        region_dfs[region] = df

if not region_dfs:
    raise RuntimeError(f"Keine CDR-Files in '{BASE_DIR}' gefunden! "
                       "Überprüfe, ob die Dateien wirklich dort liegen.")

# 2) Outer-Join aller Regionen
merged = None
for df in region_dfs.values():
    merged = df.copy() if merged is None else merged.join(df, how="outer")

# 3) Index zurück in Spalten & umbenennen
merged = (
    merged
    .reset_index()
    .rename(columns={
        "pdb_member": "PDB_ID",
        "antigen_name": "antigen_name"
    })
)

# 4) Spaltenreihenfolge
cols = ["PDB_ID", "antigen_name"] + [f"CF_{r}" for r in cdr_regions]
cols = [c for c in cols if c in merged.columns]
merged = merged[cols]

# 5) Speichern
out_fn = "data\MMseqs2/MMseqs2_summary_cluster.tsv"
merged.to_csv(out_fn, sep="\t", index=False)


#Da clustering für SEQ H2-5, Seq L3-5 nicht erstellt wurden ist kommt es zu missing values im MMseqs2_summary cluster dokument



<>:63: SyntaxWarning: invalid escape sequence '\M'
<>:63: SyntaxWarning: invalid escape sequence '\M'
C:\Users\chris\AppData\Local\Temp\ipykernel_19644\1404057441.py:63: SyntaxWarning: invalid escape sequence '\M'
  out_fn = "data\MMseqs2/MMseqs2_summary_cluster.tsv"


In [12]:
import glob
import pandas as pd

# Pfad zu deinen Cluster-Files
BASE_DIR = "MMSeqs2"
pattern = f"{BASE_DIR}/SEQ_*_clusters.tsv"

unique_pdb = set()

for fn in glob.glob(pattern):
    # nur die Spalte pdb_member einlesen
    df = pd.read_csv(fn, sep="\t", usecols=["pdb_member"])
    unique_pdb.update(df["pdb_member"].dropna().unique())

print(f"Absolute Anzahl unterschiedlicher PDB-Einträge: {len(unique_pdb)}")


Absolute Anzahl unterschiedlicher PDB-Einträge: 1198
